<a href="https://colab.research.google.com/github/rohan-sircar/ai-toolkit/blob/main/examples/colab-notebooks/colab-axolotl-example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tune Qwen3 14B with Axolotl

[<img src="https://raw.githubusercontent.com/axolotl-ai-cloud/axolotl/main/image/axolotl-badge-web.png" alt="Built with Axolotl" width="200" height="32"/>](https://github.com/axolotl-ai-cloud/axolotl)

Axolotl is the most performant LLM post-training framework available, delivering faster training with efficient, consistent and stable performance. Train your workload and ship your product 30% faster; saving you both time and money.

- ⭐ us on [GitHub](https://github.com/axolotl-ai-cloud/axolotl)
- 📜 Read the [Docs](http://docs.axolotl.ai/)
- 💬 Chat with us on [Discord](https://discord.gg/mnpEYgRUmD)
- 📰 Get updates on [X/Twitter](https://x.com/axolotl_ai)


# Installation

Axolotl is easy to install from [pip](https://pypi.org/project/axolotl/), or use our [pre-built Docker images](http://docs.axolotl.ai/docs/docker.html) for a hassle free dependency experience. See our [docs](http://docs.axolotl.ai/docs/installation.html) for more information.

In [1]:
%%capture
# This step can take ~5-10 minutes to install dependencies
!pip install --no-build-isolation "axolotl>=0.16.1"
!pip install "cut-cross-entropy[transformers] @ git+https://github.com/axolotl-ai-cloud/ml-cross-entropy.git@5effb44"

In [2]:
# Colab ships an older numpy that gets half-upgraded during install; force a
# clean, consistent numpy so it doesn't crash with a missing '_blas_supports_fpe'.
# Restart the runtime (Runtime -> Restart session) after this before importing axolotl.
!pip install --force-reinstall --no-cache-dir "numpy>=2.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 298.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.3.5
    Uninstalling numpy-2.3.5:
      Successfully uninstalled numpy-2.3.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mistral-common 1.11.0 requires numpy<2.4,>=1.25; python_version <= "3.12", but you have numpy 2.4.6 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.


## Demo: Talk Like a Pirate

In this demo, we are training the model ***to respond like a pirate***. This was chosen as a way to easily show how to train a model to respond in a certain style of your choosing (without being prompted) and is quite easy to validate within the scope of a Colab.

### Upload your own dataset or use a Huggingface dataset

You can choose to use your own JSONL file from your own [Google Drive](https://drive.google.com/drive/home); for example downloading the [Pirate-Ultrachat JSONL](https://huggingface.co/datasets/winglian/pirate-ultrachat-10k/blob/main/train.jsonl) to your Google Drive. JSONL datasets should be formatted similar to the [OpenAI dataset format](https://cookbook.openai.com/examples/chat_finetuning_data_prep).

You can also simply use the [`winglian/pirate-ultrachat-10k`](https://huggingface.co/datasets/winglian/pirate-ultrachat-10k) dataset directly.


In [ ]:
# Default to HF dataset location
dataset_id = "winglian/pirate-ultrachat-10k"
uploaded = {}

In [19]:
import os

# Optionally, upload your own JSONL to your Google Drive
GOOGLE_DRIVE_PATH = "MyDrive/datasets/starslop.chatml.jsonl"  # ex: "MyDrive/Colab\ Notebooks/train.jsonl"

# "Select All" permissions, or you may get the error:
# "MessageError: Error: credential propagation was unsuccessful"
if GOOGLE_DRIVE_PATH:
    from google.colab import drive

    # Mount your Google Drive
    GOOGLE_DRIVE_MNT = "/content/drive/"
    drive.mount(GOOGLE_DRIVE_MNT, force_remount=True)
    tmp_path = os.path.join(GOOGLE_DRIVE_MNT, GOOGLE_DRIVE_PATH.lstrip("/"))
    # make sure file exists
    if not os.path.isfile(tmp_path):
        raise ValueError(f"File {tmp_path} does not exist")
    dataset_id = tmp_path

# Configure for Supervised Fine-Tuning (SFT)

In [1]:
from axolotl.cli.config import load_cfg
from axolotl.utils.dict import DictDefault

# Axolotl provides full control and transparency over model and training configuration
config = DictDefault(
    base_model="google/gemma-4-E4B",  # Use the instruct tuned model, but we're aligning it to be a pirate
    load_in_4bit=True,  # set to True for qLoRA
    adapter="qlora",
    lora_r=32,
    lora_alpha=64,
    lora_target_modules=['model.language_model.layers.[\d]+.(_checkpoint_wrapped_module.)?(mlp|self_attn).(up|down|gate|q|k|v|o)_proj'],
    freeze_mm_modules=True,          # Freeze vision/audio encoders for text-only training
    use_gradient_checkpointing=True,
    lora_qkv_kernel=True,  # optimized triton kernels for LoRA
    lora_o_kernel=True,
    lora_mlp_kernel=True,
    embeddings_skip_upcast=True,  # keep embeddings in fp16 so the model fits in 15GB VRAM
    xformers_attention=True,  # use xformers on Colab w/ T4 for memory efficient attention
    plugins=[
        "axolotl.integrations.cut_cross_entropy.CutCrossEntropyPlugin",
    ],
    sample_packing=False,  # Disabled to prevent OOM on T4
    learning_rate=0.00019,
    sequence_len=4096,
    micro_batch_size=1,
    gradient_accumulation_steps=1,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={
        "use_reentrant": False,
    },
    optimizer="paged_adamw_8bit",
    lr_scheduler="cosine",
    warmup_steps=5,
    fp16=True,
    bf16=False,
    max_grad_norm=0.1,
    num_epochs=1,
    saves_per_epoch=2,
    logging_steps=1,
    output_dir="./outputs/gemma-e4b-sseth-1",
    chat_template="chatml",
    datasets=[
        {
            "path": "/content/drive/MyDrive/datasets/starslop.chatml.jsonl",
            "type": "chat_template",
            "split": "train",
            "eot_tokens": ["<|im_end|>"],
        }
    ],
    dataloader_prefetch_factor=2,  # Reduced to save memory
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

# validates the configuration
cfg = load_cfg(config)

<>:11: SyntaxWarning: invalid escape sequence '\d'
<>:11: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_14975/3475660449.py:11: SyntaxWarning: invalid escape sequence '\d'
  lora_target_modules=['model.language_model.layers.[\d]+.(_checkpoint_wrapped_module.)?(mlp|self_attn).(up|down|gate|q|k|v|o)_proj'],


[2026-06-16 03:08:00,043] [WARNING] [torchao] Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[2026-06-16 03:08:00,049] [WARNING] [torchao] Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
[2026-06-16 03:08:06,157] [INFO] [axolotl.integrations.base] Attempting to load plugin: axolotl.integrations.cut_cross_entropy.CutCrossEntropyPlugin
[2026-06-16 03:08:10,608] [INFO] [axolotl.integrations.base] Plugin loaded successfully: axolotl.integrations.cut_cross_entropy.CutCrossEntropyPlugin
[2026-06-16 03:08:10,998] [WARNING] [axolotl.utils.schemas.config] `xformers_attention: true` is deprecated and will be removed in a future release. Use `attn_implementation: xformers` instead.
[2026

In [15]:
from axolotl.utils import set_pytorch_cuda_alloc_conf

set_pytorch_cuda_alloc_conf()

# Datasets

Axolotl has a robust suite of loaders and transforms to parse most open datasets of any format into the appropriate chat template for your model. Axolotl will mask input tokens from the user's prompt so that the train loss is only calculated against the model's response. For more information, [see our documentation](http://docs.axolotl.ai/docs/dataset-formats/conversation.html) on dataset preparation.


In [22]:
from axolotl.common.datasets import load_datasets

# Load, parse and tokenize the datasets to be formatted with qwen3 chat template
# Drop long samples from the dataset that overflow the max sequence length
dataset_meta = load_datasets(cfg=cfg)

Generating train split: 0 examples [00:00, ? examples/s]

Tokenizing Prompts (num_proc=2):   0%|          | 0/35 [00:00<?, ? examples/s]

Dropping Invalid Sequences (<None or >4096) (num_proc=2):   0%|          | 0/35 [00:00<?, ? examples/s]

Drop Samples with Zero Trainable Tokens (num_proc=2):   0%|          | 0/35 [00:00<?, ? examples/s]

Add position_id column (Sample Packing) (num_proc=2):   0%|          | 0/35 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/35 [00:00<?, ? examples/s]

# Training



In [ ]:
from axolotl.train import train

# just train the first 25 steps for demo.
# This is sufficient to align the model as we've used packing to maximize the trainable samples per step.
cfg.max_steps = 25
model, tokenizer, trainer = train(cfg=cfg, dataset_meta=dataset_meta)

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

# Inferencing the trained model

In [ ]:
from transformers import TextStreamer

messages = [
    {
        "role": "user",
        "content": "Explain the Pythagorean theorem to me.",
    },
]

prompt = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=False,
    enable_thinking=False,
)

outputs = model.generate(
    **tokenizer(prompt, return_tensors="pt").to("cuda"),
    max_new_tokens=192,
    temperature=1.0,
    top_p=0.8,
    top_k=32,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

Ahoy there, matey! Shiver me timbers, ye be lookin' for the Pythagorean theorem, eh? Well, hold yer horses and listen up, for I'll be tellin' ye all about it in me own special way.

The Pythagorean theorem be a real gem of a mathematical trick that helps ye find the length of a side of a right triangle. Now, a right triangle be a triangle with a right angle, which be that little corner that looks like a square. 

The theorem be named after a clever fellow named Pythagoras, who be a mathematician from ancient Greece. He discovered that if ye have a right triangle, the square of the length of the hypotenuse (that be the side opposite the right angle) be equal to the sum of the squares of the other two sides. 

In other words, if ye have a triangle with sides of length a, b, and c (


# Saving your trained model

Axolotl automatically saves checkpoints to the `output_dir` path.



In [ ]:
# Show the saved checkpoints in the output_dir
!ls -lh "./outputs/qwen-sft-pirate-rrr"

total 506M
-rw-r--r-- 1 root root  845 May  7 22:21 adapter_config.json
-rw-r--r-- 1 root root 491M May  7 22:21 adapter_model.safetensors
-rw-r--r-- 1 root root  707 May  7 22:11 added_tokens.json
drwxr-xr-x 2 root root 4.0K May  7 22:17 checkpoint-13
drwxr-xr-x 2 root root 4.0K May  7 22:21 checkpoint-25
-rw-r--r-- 1 root root 1.2K May  7 22:11 config.json
-rw-r--r-- 1 root root 1.6M May  7 22:11 merges.txt
-rw-r--r-- 1 root root 2.6K May  7 22:21 README.md
-rw-r--r-- 1 root root  613 May  7 22:11 special_tokens_map.json
-rw-r--r-- 1 root root 9.5K May  7 22:11 tokenizer_config.json
-rw-r--r-- 1 root root  11M May  7 22:11 tokenizer.json
-rw-r--r-- 1 root root 2.7M May  7 22:11 vocab.json


Setting `hub_model_id: ` in the original config would have automatically uploaded the model to HuggingFace Hub (e.g. `hub_model_id: username/model_id`)

If you prefer to manually upload the training artifacts, we can still upload the entire final checkpoint to HuggingFace from the CLI.

In [ ]:
from huggingface_hub import notebook_login

# remove the partial epoch checkpoints
!rm -rf "./outputs/qwen-sft-pirate-rrr/checkpoint-*"

# HF Notebook login widget
notebook_login()

# upload the LoRA adapter for your model to HF, remember to update the username/model-name below
!huggingface-cli upload --repo-type=model winglian/pirate-qwen-14B "./outputs/qwen-sft-pirate-rrr"

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`huggingface-cli upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.
Start hashing 40 files.
Finished hashing 40 files.
Uploading files using Xet Storage..
Uploading...:  87% 1.82G/2.10G [00:23<00:04, 67.3MB/s]Cancellation requested; stopping current tasks.
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/huggingface_hub/_commit_api.py", line 598, in _upload_xet_files
    upload_files(
RuntimeError: Xet Runtime Error: Task cancelled; possible runtime shutdown in progress (task 9 was cancelled).

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/bin/huggingfa